In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Load processed data using your absolute path
df = pd.read_csv('/Users/darshanv/credit-risk-scorer/data/application_train_processed.csv')

# Drop TARGET and SK_ID_CURR (ID column — meaningless as a feature)
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Default rate: {y.mean():.2%}")

Features shape: (307511, 77)
Target shape: (307511,)
Default rate: 8.07%


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")
# Both should show ~8.07% — confirms stratification worked

Train size: (246008, 77)
Test size: (61503, 77)
Train default rate: 8.07%
Test default rate: 8.07%


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. We must scale our data for Logistic Regression because 
# features like AMT_INCOME (millions) and age_years (tens) are on vastly different scales.
# We use a Pipeline to chain the scaler and the model together cleanly.
baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(
        class_weight='balanced', # The penalty system for imbalanced data
        max_iter=1000,           # Gives the model enough time to find the best line
        random_state=42
    ))
])

# 2. Train (fit) the model on the 80% training data
print("Training Baseline Logistic Regression...")
baseline_pipeline.fit(X_train, y_train)

# 3. Ask the model to predict the probability of default on our hidden 20% test data
# predict_proba returns two columns: [Probability of 0, Probability of 1]
# We only want the probability of 1 (default), which is the second column [:, 1]
y_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

# 4. Calculate the ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Baseline ROC-AUC Score: {roc_auc:.4f}")

Training Baseline Logistic Regression...
Baseline ROC-AUC Score: 0.7475


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest Classifier (this might take a minute or two)...")

# 1. Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=100,        # Build a forest of 100 individual decision trees
    max_depth=10,            # Limit depth to 10 splits so the trees don't overfit (memorize)
    class_weight='balanced', # Keep our penalty system for the 92/8 imbalance
    random_state=42,         # Ensure we get the exact same forest every time we run thi
    n_jobs=-1                # Use all CPU cores to train faster
)

# 2. Train the model on the 80% training data
rf_model.fit(X_train, y_train)

# 3. Predict the probability of default on the hidden 20% test data
# We only want the probability of 1 (default), which is the second column [:, 1]
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# 4. Calculate the new ROC-AUC score
rf_roc_auc = roc_auc_score(y_test, rf_pred_proba)
print(f"Random Forest ROC-AUC Score: {rf_roc_auc:.4f}")


Training Random Forest Classifier (this might take a minute or two)...
Random Forest ROC-AUC Score: 0.7423


In [12]:
# Initialize our champion model
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Set up the 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("Running 5-fold cross validation...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Train and predict
    lgb_model.fit(X_fold_train, y_fold_train)
    val_preds = lgb_model.predict_proba(X_fold_val)[:, 1]
    
    # Evaluate
    score = roc_auc_score(y_fold_val, val_preds)
    cv_scores.append(score)
    print(f"  Fold {fold+1}: ROC-AUC = {score:.4f}")

print(f"\nMean ROC-AUC: {np.mean(cv_scores):.4f}")
print(f"Std deviation: {np.std(cv_scores):.4f}")

Running 5-fold cross validation...
[LightGBM] [Info] Number of positive: 15888, number of negative: 180918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4546
[LightGBM] [Info] Number of data points in the train set: 196806, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
  Fold 1: ROC-AUC = 0.7528
[LightGBM] [Info] Number of positive: 15888, number of negative: 180918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014601 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4540
[LightGBM] [Info] Number of data points in the t

In [13]:
import joblib

# Define the absolute path where the model will live
model_path = '/Users/darshanv/credit-risk-scorer/models/champion_lightgbm.pkl'

# Save the trained LightGBM model to disk
joblib.dump(lgb_model, model_path)

print(f"Success! Champion model saved to: {model_path}")

Success! Champion model saved to: /Users/darshanv/credit-risk-scorer/models/champion_lightgbm.pkl


In [14]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# 1. Define the pipeline: SMOTE first, then the LightGBM model
smote_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lgb', lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

# 2. Run the same CV process, but using our new SMOTE pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smote_cv_scores = []

print("Running 5-fold cross validation with SMOTE...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Train pipeline (SMOTE happens only on X_fold_train)
    smote_pipeline.fit(X_fold_train, y_fold_train)
    
    # Predict
    val_preds = smote_pipeline.predict_proba(X_fold_val)[:, 1]
    
    # Evaluate
    score = roc_auc_score(y_fold_val, val_preds)
    smote_cv_scores.append(score)
    print(f"  Fold {fold+1}: ROC-AUC = {score:.4f}")

print(f"\nMean ROC-AUC (with SMOTE): {np.mean(smote_cv_scores):.4f}")


Running 5-fold cross validation with SMOTE...
[LightGBM] [Info] Number of positive: 180918, number of negative: 180918
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030522 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7424
[LightGBM] [Info] Number of data points in the train set: 361836, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
  Fold 1: ROC-AUC = 0.7310
[LightGBM] [Info] Number of positive: 180918, number of negative: 180918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017471 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7425
[LightGBM] [Info] Number of data points in the train set: 361836, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -

In [15]:
# Create a summary comparison table
results_df = pd.DataFrame({
    'Model': ['Logistic Regression (Baseline)', 'LightGBM (Class Weights)', 'LightGBM (SMOTE)'],
    'Mean ROC-AUC': [0.7475, 0.7548, 0.7301]
})

print("Final Model Comparison:")
print(results_df)

Final Model Comparison:
                            Model  Mean ROC-AUC
0  Logistic Regression (Baseline)        0.7475
1        LightGBM (Class Weights)        0.7548
2                LightGBM (SMOTE)        0.7301


In [16]:
import joblib

# After CV proves the model is stable (0.7548 mean),
# we now retrain on the COMPLETE training set
# This gives the model maximum data to learn from
# before we evaluate on the locked test set

print("Retraining champion LightGBM on full training data...")

final_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# Get predictions on locked test set
test_preds_proba = final_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_preds_proba)

print(f"Final model test ROC-AUC: {test_roc_auc:.4f}")
print("This is your headline number for your resume and README")

Retraining champion LightGBM on full training data...
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015310 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4538
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 73
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Final model test ROC-AUC: 0.7597
This is your headline number for your resume and README


In [17]:
from sklearn.metrics import f1_score, average_precision_score
import pandas as pd

# Default threshold is 0.5 — but with 8% default rate
# that's often too high and misses many defaulters
# We test different thresholds and find the best one

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
results = []

for thresh in thresholds:
    preds = (test_preds_proba >= thresh).astype(int)
    f1 = f1_score(y_test, preds)
    precision = (preds[y_test == 1]).mean()
    recall = (y_test[preds == 1]).mean() if preds.sum() > 0 else 0
    results.append({
        'Threshold': thresh,
        'F1 Score': round(f1, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4)
    })

threshold_df = pd.DataFrame(results)
print(threshold_df.to_string(index=False))
print(f"\nBest F1 threshold: {threshold_df.loc[threshold_df['F1 Score'].idxmax(), 'Threshold']}")

 Threshold  F1 Score  Precision  Recall
      0.10    0.1527     0.9960  0.0827
      0.15    0.1610     0.9853  0.0876
      0.20    0.1728     0.9617  0.0949
      0.25    0.1879     0.9345  0.1044
      0.30    0.2040     0.8987  0.1151
      0.35    0.2201     0.8538  0.1263
      0.40    0.2371     0.8032  0.1391
      0.50    0.2707     0.6783  0.1691

Best F1 threshold: 0.5


In [18]:
# Overwrite the old corrupted pkl with the properly trained model
model_path = '/Users/darshanv/credit-risk-scorer/models/champion_lightgbm.pkl'
joblib.dump(final_model, model_path)
print(f"Correct champion model saved to: {model_path}")

# Also save X_train columns so Phase 6 knows exact feature order
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, '/Users/darshanv/credit-risk-scorer/models/feature_names.pkl')
print(f"Feature names saved — {len(feature_names)} features")

Correct champion model saved to: /Users/darshanv/credit-risk-scorer/models/champion_lightgbm.pkl
Feature names saved — 77 features


In [ ]:
# This is the full story of Phase 5
comparison = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Imbalance Strategy': 'Class weights',
        'ROC-AUC': 0.7475,
        'Notes': 'Single split baseline'
    },
    {
        'Model': 'LightGBM V1',
        'Imbalance Strategy': 'Class weights',
        'ROC-AUC': 0.7548,
        'Notes': '5-fold CV, std=0.0017'
    },
    {
        'Model': 'LightGBM + SMOTE',
        'Imbalance Strategy': 'SMOTE',
        'ROC-AUC': 0.7301,
        'Notes': 'Degraded — noisy synthetic data'
    },
    {
        'Model': 'LightGBM FINAL',
        'Imbalance Strategy': 'Class weights',
        'ROC-AUC': test_roc_auc,
        'Notes': 'Retrained on full train set'
    }
])

print(comparison.to_string(index=False))
print("\nChampion: LightGBM with class weights")

              Model Imbalance Strategy  ROC-AUC                           Notes
Logistic Regression      Class weights 0.747500           Single split baseline
        LightGBM V1      Class weights 0.754800           5-fold CV, std=0.0017
   LightGBM + SMOTE              SMOTE 0.730100 Degraded — noisy synthetic data
     LightGBM FINAL      Class weights 0.759728     Retrained on full train set

Champion: LightGBM with class weights
